In [1]:
%pip install -q nnunetv2 SimpleITK nibabel openneuro-py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 873.8/873.8 kB 42.2 MB/s eta 0:0

In [2]:
import os, json
from pathlib import Path

ROOT = Path("/kaggle/working/nnunet")   
BIDS = ROOT / "ds004199"
DATASET_ID = 501
DATASET_NAME = "Dataset501_BonnFCD"
TRAINER = "nnUNetTrainer_50epochs"       
CONFIG = "3d_fullres"
FOLDS = [0]                               

os.environ["nnUNet_raw"] = str(ROOT / "raw")
os.environ["nnUNet_preprocessed"] = str(ROOT / "preprocessed")
os.environ["nnUNet_results"] = str(ROOT / "results")
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path(os.environ["nnUNet_results"]) / DATASET_NAME / f"{TRAINER}__nnUNetPlans__{CONFIG}"
print("результаты будут в:", RESULTS_DIR)

результаты будут в: /kaggle/working/nnunet/results/Dataset501_BonnFCD/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres


In [3]:
import openneuro
openneuro.download(dataset="ds004199", target_dir=str(BIDS))

🌍 Preparing to download ds004199 …
📥 Checking 771 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004199 (downloaded 771 files and 3.9 GB).


In [4]:
import SimpleITK as sitk

DS = Path(os.environ["nnUNet_raw"]) / DATASET_NAME
(DS / "imagesTr").mkdir(parents=True, exist_ok=True)
(DS / "labelsTr").mkdir(parents=True, exist_ok=True)

def find(files, keys, bad=("roi", "mask", "label", "lesion")):
    for f in sorted(files):
        n = f.name.lower()
        if any(k in n for k in keys) and not any(b in n for b in bad):
            return f

def t1_to_flair(t1, flair):
    fix, mov = sitk.Cast(flair, sitk.sitkFloat32), sitk.Cast(t1, sitk.sitkFloat32)
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.1, seed=42)
    reg.SetOptimizerAsGradientDescent(1.0, 200)
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    reg.SetInitialTransform(sitk.CenteredTransformInitializer(
        fix, mov, sitk.Euler3DTransform()), inPlace=False)
    tx = reg.Execute(fix, mov)
    return sitk.Resample(mov, fix, tx, sitk.sitkLinear, 0.0, sitk.sitkFloat32)

n = 0
for sub in sorted(BIDS.glob("sub-*")):
    files = list(sub.rglob("*.nii*"))
    flair, t1 = find(files, ["flair"]), find(files, ["t1w"])
    roi = next((f for f in files if any(k in f.name.lower()
                for k in ("roi", "lesion", "mask", "label"))), None)
    if not flair or not t1:
        print("пропуск", sub.name); continue

    case = sub.name.replace("-", "")
    fl = sitk.ReadImage(str(flair))
    if roi:
        lab = sitk.Resample(sitk.ReadImage(str(roi)), fl, sitk.Transform(),
                            sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8) > 0
    else:
        lab = sitk.Image(fl.GetSize(), sitk.sitkUInt8); lab.CopyInformation(fl)

    sitk.WriteImage(sitk.Cast(fl, sitk.sitkFloat32), str(DS / "imagesTr" / f"{case}_0000.nii.gz"))
    sitk.WriteImage(t1_to_flair(sitk.ReadImage(str(t1)), fl), str(DS / "imagesTr" / f"{case}_0001.nii.gz"))
    sitk.WriteImage(sitk.Cast(lab, sitk.sitkUInt8), str(DS / "labelsTr" / f"{case}.nii.gz"))
    n += 1

json.dump({"channel_names": {"0": "FLAIR", "1": "T1"},
           "labels": {"background": 0, "lesion": 1},
           "numTraining": n, "file_ending": ".nii.gz"},
          open(DS / "dataset.json", "w"), indent=2)
print("Готово:", n)

Готово: 170


In [5]:
from nnunetv2.experiment_planning.plan_and_preprocess_api import (
    extract_fingerprints, plan_experiments, preprocess)

extract_fingerprints([DATASET_ID], check_dataset_integrity=True)
plan_experiments([DATASET_ID])
preprocess([DATASET_ID], configurations=(CONFIG,), num_processes=(4,))

Dataset501_BonnFCD
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer


Extracting dataset fingerprint: 100%|██████████| 170/170 [01:00<00:00,  2.82it/s]



############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [1.03 1.03 1.03]. 
Current patch size: (np.int64(160), np.int64(160), np.int64(96)). 
Current median shape: [248.54368932 248.54368932 155.33980583]
Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [256. 256. 160.], 3d_lowres: [249, 249, 155]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 80, 'patch_size': (np.int64(256), np.int64(160)), 'median_image_size_in_voxels': array([256., 160.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [False, False], 'resampli

Preprocessing cases: 100%|██████████| 170/170 [03:43<00:00,  1.32s/it]


In [6]:
import torch
from nnunetv2.run.run_training import run_training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("устройство:", device)
for fold in FOLDS:
    run_training(str(DATASET_ID), CONFIG, fold,
                 trainer_class_name=TRAINER, device=device)

устройство: cuda

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-10-22 22:06:18.788427: Using torch.compile...
2026-10-22 22:06:22.815974: do_dummy_2d_data_aug: False
2026-10-22 22:06:22.817277: Creating new 5-fold cross-validation split...
2026-10-22 22:06:22.819449: Desired fold for training: 0
2026-10-22 22:06:22.819608: This split ha

W1022 22:06:43.173000 23 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.13/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


2026-10-22 22:11:00.478827: train_loss 0.0116
2026-10-22 22:11:00.479452: val_loss -0.0408
2026-10-22 22:11:00.479961: Pseudo dice [np.float32(0.0)]
2026-10-22 22:11:00.480251: Epoch time: 275.23 s
2026-10-22 22:11:00.480463: Yayy! New best EMA pseudo Dice: 0.0
2026-10-22 22:11:01.893493: 
2026-10-22 22:11:01.894024: Epoch 1
2026-10-22 22:11:01.894488: Current learning rate: 0.00982
2026-10-22 22:14:07.939582: train_loss -0.0321
2026-10-22 22:14:07.940183: val_loss -0.0486
2026-10-22 22:14:07.940786: Pseudo dice [np.float32(0.0)]
2026-10-22 22:14:07.940966: Epoch time: 186.05 s
2026-10-22 22:14:09.272349: 
2026-10-22 22:14:09.272813: Epoch 2
2026-10-22 22:14:09.273109: Current learning rate: 0.00964
2026-10-22 22:17:14.466396: train_loss -0.0345
2026-10-22 22:17:14.467062: val_loss -0.0397
2026-10-22 22:17:14.467781: Pseudo dice [np.float32(0.0)]
2026-10-22 22:17:14.467963: Epoch time: 185.2 s
2026-10-22 22:17:15.863210: 
2026-10-22 22:17:15.863608: Epoch 3
2026-10-22 22:17:15.864058: 

In [7]:
import numpy as np

for fold in FOLDS:
    summary = RESULTS_DIR / f"fold_{fold}" / "validation" / "summary.json"
    if not summary.exists():
        print(f"фолд {fold}: нет файла {summary}")
        print("есть папки:", [p.name for p in (Path(os.environ['nnUNet_results']) / DATASET_NAME).glob('*')])
        continue

    s = json.load(open(summary))
    dices, det, n_pat, fp_ctrl, n_ctrl = [], 0, 0, 0, 0
    print(f"\n===== фолд {fold} =====")
    for c in s["metric_per_case"]:
        m = c["metrics"]["1"]
        name = Path(c["reference_file"]).name
        if m["TP"] + m["FN"] > 0:                 
            n_pat += 1
            found = m["TP"] > 0
            det += found
            dices.append(m["Dice"])
            print(f"{name}: Dice={m['Dice']:.2f}  {'найден' if found else 'ПРОПУЩЕН'}")
        else:                                     
            n_ctrl += 1
            fp_ctrl += m["FP"] > 0

    print(f"\nПациентов: {n_pat}, здоровых: {n_ctrl}")
    print(f"Средний Dice (только пациенты): {np.mean(dices):.3f}")
    print(f"Чувствительность (очаг найден): {det}/{n_pat} = {det/max(n_pat,1):.0%}")
    print(f"Специфичность (у здоровых ничего не найдено): "
          f"{n_ctrl-fp_ctrl}/{n_ctrl} = {(n_ctrl-fp_ctrl)/max(n_ctrl,1):.0%}")


===== фолд 0 =====
sub00003.nii.gz: Dice=0.55  найден
sub00018.nii.gz: Dice=0.13  найден
sub00034.nii.gz: Dice=0.00  ПРОПУЩЕН
sub00043.nii.gz: Dice=0.05  найден
sub00062.nii.gz: Dice=0.00  ПРОПУЩЕН
sub00068.nii.gz: Dice=0.24  найден
sub00074.nii.gz: Dice=0.00  ПРОПУЩЕН
sub00101.nii.gz: Dice=0.00  ПРОПУЩЕН
sub00112.nii.gz: Dice=0.17  найден
sub00123.nii.gz: Dice=0.49  найден
sub00142.nii.gz: Dice=0.10  найден
sub00144.nii.gz: Dice=0.06  найден
sub00146.nii.gz: Dice=0.10  найден

Пациентов: 13, здоровых: 21
Средний Dice (только пациенты): 0.145
Чувствительность (очаг найден): 9/13 = 69%
Специфичность (у здоровых ничего не найдено): 0/21 = 0%
